# Tuning the Hyperparameters of your Machine Learning Model using GridSearchCV

[Original article source](https://towardsdatascience.com/tuning-the-hyperparameters-of-your-machine-learning-model-using-gridsearchcv-7fc2bb76ff27/)

Two of the key challenges in machine learning are finding the right algorithm to use and optimizing your model. If you are familiar with machine learning, you may have worked with algorithms like Linear Regression, Logistic Regression, Decision Trees, Support Vector Machines, etc. Once you have decided on using a particular algorithm for your machine learning model, the next challenge is how to fine-tune the hyperparameters of your model so that your model works well with the dataset you have. In this article, I want to focus on the latter part – fine-tuning the hyperparameters of your model. As complex as the term may sound, fine-tuning your hyperparameters can actually be done quite easily using the **GridSearchCV** function in the `sklearn` module.

## Performing Classification using Logistic Regression

Before you learn how to fine-tune the hyperparameters of your machine learning model, let’s try to build a model using the classic Breast Cancer dataset that ships with sklearn. Since this is a classification problem, we shall use the Logistic Regression as an example.

> For classification problem, you can also use other algorithms like Support Vector Machines (SVM), K-Nearest Neighbors (KNN), Naive Bayes, and more. But for this article, I will use Logistic Regression.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer

In [2]:
bc = load_breast_cancer()

df = pd.DataFrame(bc.data, columns = bc.feature_names)
df['diagnosis'] = bc.target
df

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,diagnosis
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.30010,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.16220,0.66560,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.12380,0.18660,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.14440,0.42450,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.24140,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.20980,0.86630,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.19800,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.13740,0.20500,0.4000,0.1625,0.2364,0.07678,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564,21.56,22.39,142.00,1479.0,0.11100,0.11590,0.24390,0.13890,0.1726,0.05623,...,26.40,166.10,2027.0,0.14100,0.21130,0.4107,0.2216,0.2060,0.07115,0
565,20.13,28.25,131.20,1261.0,0.09780,0.10340,0.14400,0.09791,0.1752,0.05533,...,38.25,155.00,1731.0,0.11660,0.19220,0.3215,0.1628,0.2572,0.06637,0
566,16.60,28.08,108.30,858.1,0.08455,0.10230,0.09251,0.05302,0.1590,0.05648,...,34.12,126.70,1124.0,0.11390,0.30940,0.3403,0.1418,0.2218,0.07820,0
567,20.60,29.33,140.10,1265.0,0.11780,0.27700,0.35140,0.15200,0.2397,0.07016,...,39.42,184.60,1821.0,0.16500,0.86810,0.9387,0.2650,0.4087,0.12400,0


The first 30 columns are the various features, and the last column is the diagnosis (0 for malignant and 1 for benign). For simplicity, I will use the 30 columns for training and the last column as the target.

Ideally, you should perform feature selection to filter out those columns that exhibit collinearity and as well as columns that do not have a strong correlation with the target.

Let’s extract out the values for the features and label and save them as arrays:

In [3]:
dfX = df.iloc[:,:-1]   # Features - 30 columns
dfy = df['diagnosis']  # Label - last column

X = dfX.values
y = dfy.values

print(X.shape)   # (569, 30); 2D array
print(y.shape)   # (569,);    1D array

(569, 30)
(569,)


Split the dataset into a training set and testing set:

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y,
                                       test_size=0.25,
                                       random_state=2)

The following figure shows the use of the training and testing datasets:

![](https://towardsdatascience.com/wp-content/uploads/2021/10/1qKjPJi4lJM6eNlOhBRGtDQ.png)

Next, standardize the training and testing datasets:

In [5]:
X_train

array([[1.669e+01, 2.020e+01, 1.071e+02, ..., 8.737e-02, 4.677e-01,
        7.623e-02],
       [1.486e+01, 2.321e+01, 1.004e+02, ..., 1.727e-01, 3.000e-01,
        8.701e-02],
       [1.181e+01, 1.739e+01, 7.527e+01, ..., 4.306e-02, 3.200e-01,
        6.576e-02],
       ...,
       [1.246e+01, 1.283e+01, 7.883e+01, ..., 2.680e-02, 2.280e-01,
        7.028e-02],
       [1.234e+01, 1.227e+01, 7.894e+01, ..., 1.070e-01, 3.110e-01,
        7.592e-02],
       [1.747e+01, 2.468e+01, 1.161e+02, ..., 1.721e-01, 2.160e-01,
        9.300e-02]], shape=(426, 30))

In [6]:
X_test

array([[1.394e+01, 1.317e+01, 9.031e+01, ..., 1.015e-01, 2.160e-01,
        7.253e-02],
       [1.496e+01, 1.910e+01, 9.703e+01, ..., 1.489e-01, 2.962e-01,
        8.472e-02],
       [9.668e+00, 1.810e+01, 6.106e+01, ..., 2.500e-02, 3.057e-01,
        7.875e-02],
       ...,
       [1.108e+01, 1.471e+01, 7.021e+01, ..., 4.306e-02, 1.902e-01,
        7.313e-02],
       [9.667e+00, 1.849e+01, 6.149e+01, ..., 6.560e-02, 3.174e-01,
        8.524e-02],
       [1.346e+01, 1.875e+01, 8.744e+01, ..., 1.427e-01, 3.518e-01,
        8.665e-02]], shape=(143, 30))

In [7]:
from sklearn import preprocessing

scaler = preprocessing.StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.fit_transform(X_test)

In [8]:
X_train

array([[ 0.75745491,  0.19700816,  0.65229915, ..., -0.3968803 ,
         2.96709011, -0.42234605],
       [ 0.22900415,  0.88783781,  0.37173467, ...,  0.91214675,
         0.17260144,  0.16790015],
       [-0.65174714, -0.44791919, -0.68059152, ..., -1.07662937,
         0.50587379, -0.99561858],
       ...,
       [-0.46404604, -1.49449168, -0.53151546, ..., -1.32607013,
        -1.02717903, -0.7481313 ],
       [-0.49869855, -1.62301813, -0.52690918, ..., -0.09574118,
         0.35590123, -0.43931974],
       [ 0.98269622,  1.22521973,  1.02917681, ...,  0.90294229,
        -1.22714244,  0.49587555]], shape=(426, 30))

In [9]:
X_test

array([[-0.09965933, -1.45130003, -0.11775407, ..., -0.25653077,
        -1.12693797, -0.65725578],
       [ 0.17732828, -0.0084733 ,  0.14704585, ...,  0.45151738,
         0.07226415,  0.04415693],
       [-1.25974862, -0.25178304, -1.270343  , ..., -1.39926669,
         0.21431427, -0.29935692],
       ...,
       [-0.87631087, -1.07660304, -0.90978954, ..., -1.12949139,
        -1.5127162 , -0.62273177],
       [-1.26002018, -0.15689224, -1.25339896, ..., -0.79279508,
         0.38926021,  0.07407773],
       [-0.23000644, -0.09363171, -0.2308457 , ...,  0.35890348,
         0.9036312 ,  0.15520914]], shape=(143, 30))

The **StandardScaler** class rescales data to have a mean of 0 and a standard deviation of 1 (unit variance).

> Standardization of a dataset is a common requirement for many machine learning estimators: they might behave badly if the individual features do not more or less look like standard normally distributed data (e.g. Gaussian with 0 mean and unit variance). Source: https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html

Finally, use the **LogisticRegression** class from `sklearn` to build a model using the training set and then use the testing set to obtain the predictions for all the items in the testing set:


In [10]:
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression()
logreg.fit(X_train,y_train)

y_pred = logreg.predict(X_test)

To see how well your model is performing, obtain its accuracy:

In [11]:
from sklearn import metrics

print("Accuracy:",metrics.accuracy_score(y_test, y_pred))
# OR
print("Accuracy:",logreg.score(X_test, y_test))

Accuracy: 0.9790209790209791
Accuracy: 0.9790209790209791


## Understanding Cross Validation

To understand how to optimize your model built using the previous section using the GridSearchCV, you need to understand what is cross validation. Remember in the previous section we divided the dataset into a training set and a testing set?

![](https://towardsdatascience.com/wp-content/uploads/2021/10/1h2aPBBsEPvbUMvLYEjtLNQ.png)

The testing set is used to evaluate the performance of the model that you have trained using the training set. While this is a good way to evaluate the model, it might not give you a true indication of the performance of the model. For all you know, the data in the testing set may be skewed, and using it to evaluate the model may give a very biased result. A much better way is to divide the entire data set into k-folds (or k-parts, i.e. k-fold means divide the dataset into 10 equal parts). Out of the k-folds, use 1 fold for testing and k-1 folds for training:

![](https://towardsdatascience.com/wp-content/uploads/2021/10/18Keoub8_NLNUaw1B3ak4Mg.png)

In each iteration, record the metrics (such as accuracy, precision, etc) and at the end of all the iterations, calculate the mean of these metrics. This gives your model a good mixture of your data for training and testing, and gives a better benchmark for the performance for your model. This process of splitting your data into k-folds and using 1 fold for testing and k-1 fold for testing is known as **k-fold cross validation**.

## Using GridSearchCV for hyperparameters tuning

In our earlier example of the **LogisticRegression** class, we created an instance of the **LogisticRegression** class without passing it any initializers. Instead, we rely on the default values of the various parameters, such as:

* `penalty` – Specify the norm of the penalty.
* `C` – Inverse of regularization strength; smaller values specify stronger regularization.
* `solver` – Algorithm to use in the optimization problem.
* `max_iter` – Maximum number of iterations taken for the solvers to converge.

While it is alright in some cases to rely on the default values of these parameters (known as _hyperparameters_ in machine learning), it is always good to be able to fine-tune their values so that the algorithm works best for the type of data you have. Unfortunately, it is not not a trivial task to find the perfect combination of hyperparameters that can fit your data perfectly. This is where GridSearchCV comes in.

**GridSearchCV** is a function that is in sklearn‘s model_selection package. It allows you to specify the different values for each hyperparameter and try out all the possible combinations when fitting your model. It does the training and testing using cross validation of your dataset – hence the acronym "CV" in GridSearchCV. The end result of GridSearchCV is a set of hyperparameters that best fit your data according to the scoring metric that you want your model to optimize on.

Let’s first create the parameter grid, which is a dictionary containing all the various hyperparameters that you want to try when fitting your model:

In [12]:
from sklearn.model_selection import GridSearchCV

# Note that I have turned off the warnings as the GridSearchCV() function tends to generate quite a bit of warnings.
import warnings
warnings.filterwarnings('ignore')

# parameter grid
parameters = {
    'penalty' : ['l1','l2'],
    'C'       : np.logspace(-3,3,7),
    'solver'  : ['newton-cg', 'lbfgs', 'liblinear'],
}

logreg = LogisticRegression()
clf = GridSearchCV(logreg,                    # model
                   param_grid = parameters,   # hyperparameters
                   scoring='accuracy',        # metric for scoring
                   cv=10)

The GridSearchCV() function returns a LogisticRegression instance (in this example, based on the algorithm that you are using), which you can then train using your training set:

In [13]:
clf.fit(X_train,y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",LogisticRegression()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'C': array([1.e-03...e+02, 1.e+03]), 'penalty': ['l1', 'l2'], 'solver': ['newton-cg', 'lbfgs', ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",10
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed

Once you are done with the training, you can now print out the tuned-hyperparameters as well as the training accuracy:

In [14]:
print("Tuned Hyperparameters :", clf.best_params_)
print("Accuracy :",clf.best_score_)

Tuned Hyperparameters : {'C': np.float64(0.1), 'penalty': 'l2', 'solver': 'liblinear'}
Accuracy : 0.983499446290144


The accuracy of 0.9835 is now much better that the earlier accuracy of 0.9790.

With the values of the hyperparameters returned by the GridSearchCV() function, you can now use these values to build your model using the training dataset:

In [15]:
logreg = LogisticRegression(C = 0.1,
                            penalty = 'l2',
                            solver = 'liblinear')
logreg.fit(X_train,y_train)

y_pred = logreg.predict(X_test)
print("Accuracy:",logreg.score(X_test, y_test))

Accuracy: 0.958041958041958


The following figure summarizes what we have done:

![](https://towardsdatascience.com/wp-content/uploads/2021/10/1a4ENJEahtQsSKS3pWoaKLg.png)

Observe that:

* GridSearchCV uses the Training set and the Validation set to perform cross validation.
* Once the GridSearchCV found the values for the hyperparameters, we use the tuned parameters values to build a new model using the training set.
* With the Testing set, we can now evaluate our new model.

> The approach taken here allows us to have a metric in which we can measure the performance of our new model.

